# Suivi d'expériences avec MLflow : Iris + RandomForest

Exécutez les cellules **dans l'ordre** (Maj + Entrée).

## 1. Imports

In [11]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

## 2. Configuration de MLflow

- `set_tracking_uri` : où MLflow enregistre les résultats (une base SQLite locale `mlflow.db`).
- `set_experiment` : le nom de l'expérience qui regroupe tous nos essais.

In [12]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("iris-classification")

<Experiment: artifact_location='/Users/bertrandfillion/Library/CloudStorage/OneDrive-Personnel(2)/FORMATIONS/DATA-IA/IA/COURS/MLFLOW/mlflow-iris/mlruns/1', creation_time=1791384009739, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1791384009739, lifecycle_stage='active', name='iris-classification', tags={}, trace_location=None, workspace='default'>

## 3. Préparation des données

In [13]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Entraînement : {len(X_train)} exemples, test : {len(X_test)} exemples")

Entraînement : 120 exemples, test : 30 exemples


## 4. Une fonction qui entraîne un modèle et l'enregistre

Chaque appel crée un **run** MLflow. Dans ce run, on enregistre :
- les **paramètres** (réglages du modèle),
- les **métriques** (résultats),
- le **modèle** lui-même (artefact).

Le paramètre `skops_trusted_types` déclare comme sûr un objet interne des forêts aléatoires (`sklearn.tree._tree.Tree`). MLflow l'exige car il enregistre les modèles dans un format sécurisé ; ici, le modèle est créé par nous-mêmes.

In [14]:
def train_model(params, run_name=None):
    with mlflow.start_run(run_name=run_name):
        model = RandomForestClassifier(random_state=42, **params)
        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        f1 = f1_score(y_test, predictions, average="macro")

        mlflow.log_params(params)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_macro", f1)
        mlflow.sklearn.log_model(
            model,
            name="model",
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )

        print(f"{params} -> accuracy={accuracy:.3f}, f1={f1:.3f}")

## 5. Premier essai

In [15]:
train_model({"n_estimators": 10, "max_depth": 2}, run_name="premier-essai")

2026/10/07 17:43:35 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/bertrandfillion/Library/CloudStorage/OneDrive-Personnel(2)/FORMATIONS/DATA-IA/IA/COURS/MLFLOW/mlflow-iris
2026/10/07 17:43:35 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/bertrandfillion/Library/CloudStorage/OneDrive-Personnel(2)/FORMATIONS/DATA-IA/IA/COURS/MLFLOW/mlflow-iris
2026/10/07 17:43:35 INFO mlflow.utils.environment: Detected uv project at /Users/bertrandfillion/Library/CloudStorage/OneDrive-Personnel(2)/FORMATIONS/DATA-IA/IA/COURS/MLFLOW/mlflow-iris. Attempting to export requirements via 'uv export'.
2026/10/07 17:43:35 INFO mlflow.utils.uv_utils: Exported 90 dependencies via uv
2026/10/07 17:43:35 INFO mlflow.utils.environment: Successfully exported 90 requirements from uv project. Skipping package capture based inference.
2026/10/07 17:43:35 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``

{'n_estimators': 10, 'max_depth': 2} -> accuracy=1.000, f1=1.000


## 6. Autres essais à comparer

On fait varier les hyperparamètres pour avoir plusieurs runs à comparer dans l'interface.

In [16]:
trials = [
    {"n_estimators": 50, "max_depth": 4},
    {"n_estimators": 100, "max_depth": None},
]

for params in trials:
    train_model(params)

2026/10/07 17:43:35 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/bertrandfillion/Library/CloudStorage/OneDrive-Personnel(2)/FORMATIONS/DATA-IA/IA/COURS/MLFLOW/mlflow-iris
2026/10/07 17:43:35 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/bertrandfillion/Library/CloudStorage/OneDrive-Personnel(2)/FORMATIONS/DATA-IA/IA/COURS/MLFLOW/mlflow-iris
2026/10/07 17:43:35 INFO mlflow.utils.environment: Detected uv project at /Users/bertrandfillion/Library/CloudStorage/OneDrive-Personnel(2)/FORMATIONS/DATA-IA/IA/COURS/MLFLOW/mlflow-iris. Attempting to export requirements via 'uv export'.
2026/10/07 17:43:35 INFO mlflow.utils.uv_utils: Exported 90 dependencies via uv
2026/10/07 17:43:35 INFO mlflow.utils.environment: Successfully exported 90 requirements from uv project. Skipping package capture based inference.
2026/10/07 17:43:36 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``

{'n_estimators': 50, 'max_depth': 4} -> accuracy=1.000, f1=1.000


2026/10/07 17:43:36 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/bertrandfillion/Library/CloudStorage/OneDrive-Personnel(2)/FORMATIONS/DATA-IA/IA/COURS/MLFLOW/mlflow-iris
2026/10/07 17:43:36 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/bertrandfillion/Library/CloudStorage/OneDrive-Personnel(2)/FORMATIONS/DATA-IA/IA/COURS/MLFLOW/mlflow-iris
2026/10/07 17:43:36 INFO mlflow.utils.environment: Detected uv project at /Users/bertrandfillion/Library/CloudStorage/OneDrive-Personnel(2)/FORMATIONS/DATA-IA/IA/COURS/MLFLOW/mlflow-iris. Attempting to export requirements via 'uv export'.
2026/10/07 17:43:36 INFO mlflow.utils.uv_utils: Exported 90 dependencies via uv
2026/10/07 17:43:36 INFO mlflow.utils.environment: Successfully exported 90 requirements from uv project. Skipping package capture based inference.
2026/10/07 17:43:36 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``

{'n_estimators': 100, 'max_depth': None} -> accuracy=1.000, f1=1.000


## 7. Et maintenant ?

Ouvrez un terminal dans le dossier du projet et lancez :

```bash
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Puis visitez http://127.0.0.1:5000